# Combined Weather, Price and Holiday Analysis

Run cells from top to bottom. This notebook merges the three feature sources, checks data quality, summarizes delivery outcomes, and exports **plain CSV files**.

Each timezone scenario contains the same orders and is analyzed separately. Never stack the scenarios as independent orders. Weather is from the previous complete UTC day; publication latency and the source timezone remain unverified.

All orders and missing values are retained in the exported datasets. Descriptive comparisons use labeled orders only. No hypothesis tests or machine-learning models are fitted here.


## 1. Load the three datasets
Weather input may be CSV or CSV.GZ. All outputs are CSV. Run from the repository or its notebooks folder.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data/processed/research_data_price_product.csv").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project repository.")

weather_path = ROOT / "data/external/inmet/processed/order_weather_purchase_scenarios.csv"
if not weather_path.exists():
    weather_path = weather_path.with_suffix(".csv.gz")

paths = {
    "weather": weather_path,
    "price": ROOT / "data/processed/research_data_price_product.csv",
    "holiday": ROOT / "data/processed/order_holiday_features.csv",
}
tables = {
    name: pd.read_csv(path, dtype={"order_id": "string"})
    for name, path in paths.items()
}
for name, df in tables.items():
    print(f"{name}: {len(df):,} rows, {len(df.columns)} columns")


weather: 198,882 rows, 30 columns
price: 99,441 rows, 44 columns
holiday: 99,441 rows, 4 columns


## 2. Merge separately by timezone
Check unique keys, order coverage, and agreement of shared labels before merging.

In [2]:
# Purpose: Merge price/product, holiday, and pre-purchase weather
# features separately for each timezone scenario.

price = tables["price"].copy()
holiday = tables["holiday"].copy()
weather = tables["weather"].copy()

# 1. Validate merge keys.
for name, df in [("price", price), ("holiday", holiday)]:
    assert df["order_id"].notna().all(), f"{name}: missing order_id"
    assert df["order_id"].is_unique, f"{name}: duplicate order_id"

assert weather["order_id"].notna().all(), "Weather: missing order_id"
assert weather["timezone_scenario"].notna().all(), "Missing timezone scenario"

assert not weather.duplicated(
    ["order_id", "timezone_scenario"]
).any(), "Duplicate order within the same timezone scenario"

assert set(price["order_id"]) == set(holiday["order_id"]), (
    "Order IDs differ between price and holiday tables"
)
assert set(price["order_id"]) == set(weather["order_id"]), (
    "Order IDs differ between price and weather tables"
)

# 2. Merge holiday features into the base table.
base = price.merge(
    holiday,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# 3. Merge weather features separately for each timezone scenario.
combined_tables = {}

for scenario, weather_part in weather.groupby("timezone_scenario"):
    assert set(weather_part["order_id"]) == set(base["order_id"]), (
        f"{scenario}: incomplete order coverage"
    )

    # Verify shared columns before removing them from the weather table.
    check = base[["order_id", "customer_state", "is_late"]].merge(
        weather_part[["order_id", "customer_state", "is_late"]],
        on="order_id",
        validate="one_to_one",
        suffixes=("_base", "_weather")
    )

    for col in ["customer_state", "is_late"]:
        left = check[f"{col}_base"]
        right = check[f"{col}_weather"]

        same = (
            left.eq(right).fillna(False)
            | (left.isna() & right.isna())
        )
        assert same.all(), f"{scenario}: inconsistent values in {col}"

    # Keep customer state and the delivery label from the base table.
    weather_features = weather_part.drop(
        columns=["customer_state", "is_late"]
    )

    overlap = (
        set(base.columns) & set(weather_features.columns)
    ) - {"order_id"}
    assert not overlap, f"Unresolved overlapping columns: {overlap}"

    combined = base.merge(
        weather_features,
        on="order_id",
        how="left",
        validate="one_to_one"
    )

    assert len(combined) == len(price), "Row count changed after merging"
    assert combined["order_id"].is_unique, "Duplicate orders after merging"

    combined_tables[scenario] = combined

    print(f"\nTimezone scenario: {scenario}")
    print("Dataset shape:", combined.shape)
    print("Unique orders:", combined["order_id"].nunique())
    print("Delivery label counts (including missing labels):")
    print(combined["is_late"].value_counts(dropna=False))

print("\nAll merge checks passed. No files have been saved yet.")

# These tables retain retrospective columns for EDA.
# Before purchase-time modeling, explicitly select eligible predictors
# and exclude post-purchase information and the target label.


Timezone scenario: assume_sao_paulo
Dataset shape: (99441, 74)
Unique orders: 99441
Delivery label counts (including missing labels):
is_late
0.0    89936
1.0     6534
NaN     2971
Name: count, dtype: int64

Timezone scenario: assume_utc
Dataset shape: (99441, 74)
Unique orders: 99441
Delivery label counts (including missing labels):
is_late
0.0    89936
1.0     6534
NaN     2971
Name: count, dtype: int64

All merge checks passed. No files have been saved yet.


## 3. Check missing values
The feature-level percentages use all orders. Complete weather coverage is calculated on labeled orders.

In [3]:
# Purpose: Audit missingness in the added weather, holiday,
# and price/product features for each timezone scenario.

holiday_cols = [
    "purchase_is_national_holiday",
    "days_to_next_national_holiday",
    "national_holiday_count_in_estimated_window",
]

price_product_cols = [
    "order_total_value",
    "freight_to_items_ratio",
    "mean_item_value",
    "freight_per_item",
    "total_product_weight_g",
    "total_product_volume_cm3",
    "product_category_count",
    "weight_coverage_ratio",
    "volume_coverage_ratio",
    "category_coverage_ratio",
]

missingness_tables = {}

for scenario, df in combined_tables.items():
    # Select weather measurements, excluding seller coverage indicators.
    weather_cols = [
        col for col in df.columns
        if col.startswith(("origin_", "destination_"))
        and "_state_mean_" in col
        and not col.endswith("_seller_coverage")
    ]

    # Include means of station-level temperature and gust extremes.
    weather_cols += [
        col for col in df.columns
        if col.startswith(("origin_", "destination_"))
        and "_station_mean_" in col
        and not col.endswith("_seller_coverage")
        and col not in weather_cols
    ]

    assert len(weather_cols) == 14, (
        f"{scenario}: expected 14 weather measurements, "
        f"found {len(weather_cols)}"
    )

    groups = {
        "weather": weather_cols,
        "holiday": holiday_cols,
        "price_product": price_product_cols,
    }

    records = []
    for group, columns in groups.items():
        for col in columns:
            records.append({
                "feature_group": group,
                "feature": col,
                "missing_count": int(df[col].isna().sum()),
                "missing_pct": df[col].isna().mean() * 100,
            })

    report = pd.DataFrame(records).sort_values(
        ["feature_group", "missing_pct", "feature"],
        ascending=[True, False, True]
    )
    missingness_tables[scenario] = report

    labeled = df.loc[df["is_late"].notna()]
    complete_weather = labeled[weather_cols].notna().all(axis=1)

    print(f"\nTimezone scenario: {scenario}")
    print(f"Labeled orders: {len(labeled):,}")
    print(f"Late-delivery rate: {labeled['is_late'].mean():.2%}")
    print(
        "Complete weather coverage among labeled orders: "
        f"{complete_weather.mean():.2%}"
    )

    display(report.round({"missing_pct": 3}))

print("\nMissingness audit completed. No rows were removed.")


Timezone scenario: assume_sao_paulo
Labeled orders: 96,470
Late-delivery rate: 6.77%
Complete weather coverage among labeled orders: 98.73%


,feature_group,feature,missing_count,missing_pct
15,holiday,days_to_next_national_holiday,0,0.000
16,holiday,national_holiday_count_in_estimated_window,0,0.000
14,holiday,purchase_is_national_holiday,0,0.000
23,price_product,product_category_count,2226,2.239
21,price_product,total_product_weight_g,797,0.801
22,price_product,total_product_volume_cm3,791,0.795
26,price_product,category_coverage_ratio,775,0.779
20,price_product,freight_per_item,775,0.779
18,price_product,freight_to_items_ratio,775,0.779
19,price_product,mean_item_value,775,0.779



Timezone scenario: assume_utc
Labeled orders: 96,470
Late-delivery rate: 6.77%
Complete weather coverage among labeled orders: 98.73%


,feature_group,feature,missing_count,missing_pct
15,holiday,days_to_next_national_holiday,0,0.000
16,holiday,national_holiday_count_in_estimated_window,0,0.000
14,holiday,purchase_is_national_holiday,0,0.000
23,price_product,product_category_count,2226,2.239
21,price_product,total_product_weight_g,797,0.801
22,price_product,total_product_volume_cm3,791,0.795
26,price_product,category_coverage_ratio,775,0.779
20,price_product,freight_per_item,775,0.779
18,price_product,freight_to_items_ratio,775,0.779
19,price_product,mean_item_value,775,0.779



Missingness audit completed. No rows were removed.


In [4]:
# Purpose: Compare late-delivery rates between orders with
# complete and incomplete features within each feature group.
# These are descriptive comparisons, not hypothesis tests.

missingness_comparisons = []

for scenario, df in combined_tables.items():
    labeled = df.loc[df["is_late"].notna()].copy()
    feature_report = missingness_tables[scenario]

    for group in ["weather", "holiday", "price_product"]:
        columns = feature_report.loc[
            feature_report["feature_group"].eq(group), "feature"
        ].tolist()

        has_missing = labeled[columns].isna().any(axis=1)

        summary = (
            labeled.assign(
                feature_completeness=has_missing.map({
                    False: "complete",
                    True: "incomplete",
                })
            )
            .groupby("feature_completeness")
            .agg(
                order_count=("order_id", "size"),
                late_count=("is_late", "sum"),
                late_rate=("is_late", "mean"),
            )
            .reset_index()
        )

        summary.insert(0, "feature_group", group)
        summary.insert(0, "timezone_scenario", scenario)
        summary["share_of_labeled_orders_pct"] = (
            summary["order_count"] / len(labeled) * 100
        )
        summary["late_rate_pct"] = summary.pop("late_rate") * 100
        summary["late_count"] = summary["late_count"].astype(int)

        missingness_comparisons.append(summary)

missingness_comparison = pd.concat(
    missingness_comparisons, ignore_index=True
)

display(
    missingness_comparison.round({
        "share_of_labeled_orders_pct": 3,
        "late_rate_pct": 3,
    })
)

print("Only labeled orders were used.")
print("Incomplete means at least one selected feature is missing.")
print("No missing values were imputed and no files were saved.")

,timezone_scenario,feature_group,feature_completeness,order_count,late_count,share_of_labeled_orders_pct,late_rate_pct
0,assume_sao_paulo,weather,complete,95249,6449,98.734,6.771
1,assume_sao_paulo,weather,incomplete,1221,85,1.266,6.962
2,assume_sao_paulo,holiday,complete,96470,6534,100.0,6.773
3,assume_sao_paulo,price_product,complete,95071,6433,98.55,6.767
4,assume_sao_paulo,price_product,incomplete,1399,101,1.45,7.219
5,assume_utc,weather,complete,95248,6452,98.733,6.774
6,assume_utc,weather,incomplete,1222,82,1.267,6.710
7,assume_utc,holiday,complete,96470,6534,100.0,6.773
8,assume_utc,price_product,complete,95071,6433,98.55,6.767
9,assume_utc,price_product,incomplete,1399,101,1.45,7.219


Only labeled orders were used.
Incomplete means at least one selected feature is missing.
No missing values were imputed and no files were saved.


## 4. Holiday analysis
Use one scenario because holiday features and labels are shared. Use `estimated_window_days` for calendar-day delivery windows. Preserve `promised_days` unchanged; its different definition or cleaning rule still needs investigation.

Holiday exposure may be related to calendar month and delivery-window length. These comparisons do not isolate a causal holiday effect.


In [5]:
# Purpose: Describe late-delivery rates by holiday-related features.
# Holiday features and labels are identical across timezone scenarios,
# so this analysis uses one scenario only.

holiday_eda = combined_tables["assume_sao_paulo"].loc[
    lambda df: df["is_late"].notna()
].copy()

holiday_eda["purchase_holiday_group"] = (
    holiday_eda["purchase_is_national_holiday"].map({
        0: "Non-holiday purchase",
        1: "National holiday purchase",
    })
)

holiday_eda["days_to_next_holiday_group"] = pd.cut(
    holiday_eda["days_to_next_national_holiday"],
    bins=[-1, 0, 3, 7, 14, 30, float("inf")],
    labels=["0 days", "1–3 days", "4–7 days",
            "8–14 days", "15–30 days", "Over 30 days"]
)

holiday_eda["estimated_window_holiday_group"] = pd.cut(
    holiday_eda["national_holiday_count_in_estimated_window"],
    bins=[-1, 0, 1, 2, float("inf")],
    labels=["0 holidays", "1 holiday", "2 holidays", "3+ holidays"]
)

holiday_summary_tables = {}

for group_col in [
    "purchase_holiday_group",
    "days_to_next_holiday_group",
    "estimated_window_holiday_group",
]:
    assert holiday_eda[group_col].notna().all(), (
        f"Missing or unclassified values in {group_col}"
    )

    summary = (
        holiday_eda.groupby(group_col, observed=True)
        .agg(
            order_count=("order_id", "size"),
            late_count=("is_late", "sum"),
            late_rate=("is_late", "mean"),
        )
        .reset_index()
    )

    summary["late_count"] = summary["late_count"].astype(int)
    summary["late_rate_pct"] = summary.pop("late_rate") * 100
    summary["share_of_orders_pct"] = (
        summary["order_count"] / len(holiday_eda) * 100
    )

    holiday_summary_tables[group_col] = summary

    print(f"\nGrouping variable: {group_col}")
    display(summary.round({
        "late_rate_pct": 3,
        "share_of_orders_pct": 3,
    }))

print("\nDescriptive comparisons only; no hypothesis tests were performed.")


Grouping variable: purchase_holiday_group


,purchase_holiday_group,order_count,late_count,late_rate_pct,share_of_orders_pct
0,National holiday purchase,1477,72,4.875,1.531
1,Non-holiday purchase,94993,6462,6.803,98.469



Grouping variable: days_to_next_holiday_group


,days_to_next_holiday_group,order_count,late_count,late_rate_pct,share_of_orders_pct
0,0 days,1477,72,4.875,1.531
1,1–3 days,5158,311,6.029,5.347
2,4–7 days,7081,354,4.999,7.34
3,8–14 days,9818,562,5.724,10.177
4,15–30 days,20615,2159,10.473,21.369
5,Over 30 days,52321,3076,5.879,54.236



Grouping variable: estimated_window_holiday_group


,estimated_window_holiday_group,order_count,late_count,late_rate_pct,share_of_orders_pct
0,0 holidays,63733,4266,6.694,66.065
1,1 holiday,21360,1551,7.261,22.142
2,2 holidays,10054,641,6.376,10.422
3,3+ holidays,1323,76,5.745,1.371



Descriptive comparisons only; no hypothesis tests were performed.


In [6]:
# Purpose: Examine calendar composition and promised delivery time
# across holiday groups. These summaries are descriptive.

holiday_eda["purchase_year_month"] = pd.to_datetime(
    holiday_eda["order_purchase_timestamp"],
    errors="raise"
).dt.to_period("M").astype(str)

# 1. Compare promised delivery time across holiday groups.
for group_col in [
    "purchase_holiday_group",
    "days_to_next_holiday_group",
    "estimated_window_holiday_group",
]:
    summary = (
        holiday_eda.groupby(group_col, observed=True)
        .agg(
            order_count=("order_id", "size"),
            valid_window_count=("estimated_window_days", "count"),
            mean_window_days=("estimated_window_days", "mean"),
            median_window_days=("estimated_window_days", "median"),
            late_rate=("is_late", "mean"),
        )
        .reset_index()
    )

    summary["late_rate_pct"] = summary.pop("late_rate") * 100

    print(f"\nCalendar-day delivery window by {group_col}")
    display(summary.round(3))

# 2. Compare the 15–30-day group with other groups within each month.
holiday_eda["next_holiday_comparison"] = (
    holiday_eda["days_to_next_holiday_group"]
    .eq("15–30 days")
    .map({
        True: "15–30 days",
        False: "Other intervals",
    })
)

monthly_holiday_summary = (
    holiday_eda.groupby(
        ["purchase_year_month", "next_holiday_comparison"],
        observed=True
    )
    .agg(
        order_count=("order_id", "size"),
        late_count=("is_late", "sum"),
        late_rate=("is_late", "mean"),
        mean_window_days=("estimated_window_days", "mean"),
    )
    .reset_index()
)

monthly_holiday_summary["late_count"] = (
    monthly_holiday_summary["late_count"].astype(int)
)
monthly_holiday_summary["late_rate_pct"] = (
    monthly_holiday_summary.pop("late_rate") * 100
)

print("\nMonthly comparison: 15–30 days versus other intervals")
display(monthly_holiday_summary.round(3))

print(
    "\nSmall groups may have unstable rates. "
    "Months containing only one comparison group cannot "
    "support a within-month comparison."
)


Calendar-day delivery window by purchase_holiday_group


,purchase_holiday_group,order_count,valid_window_count,mean_window_days,median_window_days,late_rate_pct
0,National holiday purchase,1477,1477,25.369,25.0,4.875
1,Non-holiday purchase,94993,94993,24.357,24.0,6.803



Calendar-day delivery window by days_to_next_holiday_group


,days_to_next_holiday_group,order_count,valid_window_count,mean_window_days,median_window_days,late_rate_pct
0,0 days,1477,1477,25.369,25.0,4.875
1,1–3 days,5158,5158,24.971,24.0,6.029
2,4–7 days,7081,7081,26.116,25.0,4.999
3,8–14 days,9818,9818,24.391,24.0,5.724
4,15–30 days,20615,20615,22.644,22.0,10.473
5,Over 30 days,52321,52321,24.728,24.0,5.879



Calendar-day delivery window by estimated_window_holiday_group


,estimated_window_holiday_group,order_count,valid_window_count,mean_window_days,median_window_days,late_rate_pct
0,0 holidays,63733,63733,23.125,23.0,6.694
1,1 holiday,21360,21360,24.756,24.0,7.261
2,2 holidays,10054,10054,29.437,29.0,6.376
3,3+ holidays,1323,1323,39.788,36.0,5.745



Monthly comparison: 15–30 days versus other intervals


,purchase_year_month,next_holiday_comparison,order_count,late_count,mean_window_days,late_rate_pct
0,2016-09,15–30 days,1,1,19.000,100.000
1,2016-10,Other intervals,265,2,56.309,0.755
2,2016-12,Other intervals,1,0,27.000,0.000
3,2017-01,Other intervals,750,22,40.161,2.933
4,2017-02,Other intervals,1653,49,32.495,2.964
5,2017-03,15–30 days,1363,64,25.843,4.696
6,2017-03,Other intervals,1183,52,24.828,4.396
7,2017-04,Other intervals,2303,151,27.995,6.557
8,2017-05,Other intervals,3545,106,24.925,2.990
9,2017-06,Other intervals,3135,95,24.657,3.030



Small groups may have unstable rates. Months containing only one comparison group cannot support a within-month comparison.


In [7]:
# Purpose: Compare holiday exposure within similar estimated
# delivery-window lengths. Descriptive analysis only.

holiday_eda["delivery_window_group"] = pd.cut(
    holiday_eda["estimated_window_days"],
    bins=[-float("inf"), 0, 7, 14, 21, 28, 35, float("inf")],
    labels=[
        "0 or fewer days", "1–7 days", "8–14 days",
        "15–21 days", "22–28 days", "29–35 days", "36+ days"
    ]
)

holiday_eda["holiday_in_window"] = (
    holiday_eda["national_holiday_count_in_estimated_window"]
    .gt(0)
    .map({False: "No holidays", True: "At least one holiday"})
)

window_holiday_summary = (
    holiday_eda.groupby(
        ["delivery_window_group", "holiday_in_window"],
        observed=True
    )
    .agg(
        order_count=("order_id", "size"),
        late_count=("is_late", "sum"),
        late_rate=("is_late", "mean"),
        mean_window_days=("estimated_window_days", "mean"),
    )
    .reset_index()
)

window_holiday_summary["late_count"] = (
    window_holiday_summary["late_count"].astype(int)
)
window_holiday_summary["late_rate_pct"] = (
    window_holiday_summary.pop("late_rate") * 100
)

display(window_holiday_summary.round(3))

print(
    "Comparisons account for broad delivery-window groups only. "
    "Purchase month, geography, and other characteristics "
    "may still differ between groups."
)

,delivery_window_group,holiday_in_window,order_count,late_count,mean_window_days,late_rate_pct
0,1–7 days,No holidays,1260,213,5.305,16.905
1,8–14 days,At least one holiday,1499,59,12.747,3.936
2,8–14 days,No holidays,9243,473,11.741,5.117
3,15–21 days,At least one holiday,6324,403,18.955,6.373
4,15–21 days,No holidays,16980,1327,18.390,7.815
5,22–28 days,At least one holiday,13383,994,24.972,7.427
6,22–28 days,No holidays,21464,1655,24.632,7.711
7,29–35 days,At least one holiday,7949,627,31.594,7.888
8,29–35 days,No holidays,9790,481,31.544,4.913
9,36+ days,At least one holiday,3582,185,42.733,5.165


Comparisons account for broad delivery-window groups only. Purchase month, geography, and other characteristics may still differ between groups.


## 5. Price and product analysis
Compare means, medians and quartiles, then late rates across feature quartiles. EDA quartile cutoffs are not training preprocessing.

In [8]:
# Purpose: Summarize price/product features by delivery outcome.
# Use one timezone scenario because these features are shared.
# Preserve missing values and report valid observations.

price_eda = combined_tables["assume_sao_paulo"].loc[
    lambda df: df["is_late"].notna()
].copy()

price_features = [
    "order_total_value",
    "mean_item_value",
    "freight_value",
    "freight_to_items_ratio",
    "freight_per_item",
    "total_product_weight_g",
    "total_product_volume_cm3",
    "product_category_count",
]

price_summary_records = []

for feature in price_features:
    for label, group in price_eda.groupby("is_late"):
        values = group[feature].dropna()

        price_summary_records.append({
            "feature": feature,
            "delivery_outcome": "Late" if label == 1 else "Not late",
            "valid_count": len(values),
            "missing_count": int(group[feature].isna().sum()),
            "mean": values.mean(),
            "median": values.median(),
            "q25": values.quantile(0.25),
            "q75": values.quantile(0.75),
        })

price_summary = pd.DataFrame(price_summary_records)

display(price_summary.round(3))
print("Descriptive comparisons only. No values were imputed.")

,feature,delivery_outcome,valid_count,missing_count,mean,median,q25,q75
0,order_total_value,Not late,89936,0,158.641,104.590,61.590,175.342
1,order_total_value,Late,6534,0,176.139,114.095,67.508,190.570
2,mean_item_value,Not late,89936,0,124.012,78.990,40.900,139.000
3,mean_item_value,Late,6534,0,142.019,86.900,48.000,150.000
4,freight_value,Not late,89936,0,22.607,17.060,13.710,23.850
5,freight_value,Late,6534,0,25.248,18.230,15.100,26.900
6,freight_to_items_ratio,Not late,89936,0,0.308,0.224,0.132,0.380
7,freight_to_items_ratio,Late,6534,0,0.318,0.228,0.132,0.387
8,freight_per_item,Not late,89936,0,19.944,16.250,13.160,21.032
9,freight_per_item,Late,6534,0,22.968,17.740,15.100,23.468


Descriptive comparisons only. No values were imputed.


In [9]:
# Purpose: Describe late-delivery rates across feature quartiles.
# Quartiles are calculated on labeled orders for exploratory analysis.
# These cutoffs must not be reused as train-fitted preprocessing.

quartile_features = [
    "order_total_value",
    "freight_value",
    "total_product_weight_g",
    "total_product_volume_cm3",
]

price_quartile_tables = {}

for feature in quartile_features:
    subset = price_eda.loc[
        price_eda[feature].notna(),
        ["order_id", "is_late", feature]
    ].copy()

    subset["feature_band"] = pd.qcut(
        subset[feature],
        q=4,
        duplicates="drop"
    )

    summary = (
        subset.groupby("feature_band", observed=True)
        .agg(
            order_count=("order_id", "size"),
            late_count=("is_late", "sum"),
            feature_min=(feature, "min"),
            feature_max=(feature, "max"),
            late_rate=("is_late", "mean"),
        )
        .reset_index()
    )

    summary["late_count"] = summary["late_count"].astype(int)
    summary["late_rate_pct"] = summary.pop("late_rate") * 100
    price_quartile_tables[feature] = summary

    print(f"\nFeature: {feature}")
    display(summary.round(3))

print("\nDescriptive associations only; no causal effects were estimated.")


Feature: order_total_value


,feature_band,order_count,late_count,feature_min,feature_max,late_rate_pct
0,"(9.589, 61.85]",24125,1384,9.59,61.85,5.737
1,"(61.85, 105.28]",24173,1609,61.86,105.28,6.656
2,"(105.28, 176.26]",24056,1689,105.29,176.26,7.021
3,"(176.26, 13664.08]",24116,1852,176.30,13664.08,7.680



Feature: freight_value


,feature_band,order_count,late_count,feature_min,feature_max,late_rate_pct
0,"(-0.001, 13.85]",24153,1063,0.00,13.85,4.401
1,"(13.85, 17.17]",24091,1686,13.86,17.17,6.998
2,"(17.17, 24.02]",24112,1865,17.18,24.02,7.735
3,"(24.02, 1794.96]",24114,1920,24.03,1794.96,7.962



Feature: total_product_weight_g


,feature_band,order_count,late_count,feature_min,feature_max,late_rate_pct
0,"(1.999, 300.0]",25199,1567,2.0,300.0,6.219
1,"(300.0, 750.0]",23577,1584,302.0,750.0,6.718
2,"(750.0, 2050.0]",23584,1592,752.0,2050.0,6.750
3,"(2050.0, 184400.0]",24088,1791,2051.0,184400.0,7.435



Feature: total_product_volume_cm3


,feature_band,order_count,late_count,feature_min,feature_max,late_rate_pct
0,"(167.999, 2964.0]",24273,1637,168.0,2964.0,6.744
1,"(2964.0, 7254.0]",23956,1573,2970.0,7254.0,6.566
2,"(7254.0, 19866.0]",24129,1583,7260.0,19866.0,6.561
3,"(19866.0, 1476000.0]",24096,1741,19872.0,1476000.0,7.225



Descriptive associations only; no causal effects were estimated.


## 6. Weather analysis
Compare all 14 measurements under each timezone assumption. State averages do not measure conditions along the delivery route. Orders sharing state-day weather can be dependent.

In [10]:
# Purpose: Compare pre-purchase weather measurements by delivery
# outcome, separately for each timezone scenario.
# Missing measurements are excluded only from each feature summary.

weather_summary_records = []

for scenario, df in combined_tables.items():
    labeled = df.loc[df["is_late"].notna()]

    weather_features = missingness_tables[scenario].loc[
        lambda report: report["feature_group"].eq("weather"),
        "feature"
    ].tolist()

    for feature in weather_features:
        for label, group in labeled.groupby("is_late"):
            values = group[feature].dropna()

            weather_summary_records.append({
                "timezone_scenario": scenario,
                "feature": feature,
                "delivery_outcome": "Late" if label == 1 else "Not late",
                "valid_count": len(values),
                "missing_count": int(group[feature].isna().sum()),
                "mean": values.mean(),
                "median": values.median(),
                "q25": values.quantile(0.25),
                "q75": values.quantile(0.75),
            })

weather_summary = pd.DataFrame(weather_summary_records)

for scenario in combined_tables:
    print(f"\nTimezone scenario: {scenario}")
    display(
        weather_summary.loc[
            weather_summary["timezone_scenario"].eq(scenario)
        ].round(3)
    )

print(
    "\nScenarios were analyzed separately. "
    "These summaries do not establish causality or predictive value."
)


Timezone scenario: assume_sao_paulo


,timezone_scenario,feature,delivery_outcome,valid_count,missing_count,mean,median,q25,q75
0,assume_sao_paulo,destination_rain_state_mean_mm,Not late,89080,856,3.532,0.816,0.062,4.519
1,assume_sao_paulo,destination_rain_state_mean_mm,Late,6462,72,4.842,2.235,0.282,6.822
2,assume_sao_paulo,origin_rain_state_mean_mm,Not late,89776,160,3.537,0.812,0.062,4.704
3,assume_sao_paulo,origin_rain_state_mean_mm,Late,6526,8,4.642,2.303,0.329,7.037
4,assume_sao_paulo,origin_humidity_state_mean_pct,Not late,89778,158,72.625,73.328,66.966,79.625
5,assume_sao_paulo,origin_humidity_state_mean_pct,Late,6528,6,74.807,76.065,70.783,80.830
6,assume_sao_paulo,origin_wind_gust_max_station_mean_ms,Not late,89790,146,8.488,8.275,7.385,9.453
7,assume_sao_paulo,origin_wind_gust_max_station_mean_ms,Late,6528,6,8.552,8.343,7.538,9.497
8,assume_sao_paulo,origin_wind_speed_sample_state_mean_ms,Not late,89792,144,1.759,1.691,1.406,2.012
9,assume_sao_paulo,origin_wind_speed_sample_state_mean_ms,Late,6528,6,1.674,1.612,1.361,1.933



Timezone scenario: assume_utc


,timezone_scenario,feature,delivery_outcome,valid_count,missing_count,mean,median,q25,q75
28,assume_utc,destination_rain_state_mean_mm,Not late,89079,857,3.534,0.814,0.062,4.520
29,assume_utc,destination_rain_state_mean_mm,Late,6465,69,4.848,2.220,0.280,6.806
30,assume_utc,origin_rain_state_mean_mm,Not late,89784,152,3.552,0.812,0.061,4.705
31,assume_utc,origin_rain_state_mean_mm,Late,6527,7,4.662,2.303,0.339,7.037
32,assume_utc,origin_humidity_state_mean_pct,Not late,89788,148,72.637,73.331,66.966,79.657
33,assume_utc,origin_humidity_state_mean_pct,Late,6529,5,74.853,76.065,70.877,80.842
34,assume_utc,origin_wind_gust_max_station_mean_ms,Not late,89797,139,8.494,8.279,7.389,9.453
35,assume_utc,origin_wind_gust_max_station_mean_ms,Late,6529,5,8.553,8.335,7.544,9.483
36,assume_utc,origin_wind_speed_sample_state_mean_ms,Not late,89799,137,1.761,1.691,1.407,2.012
37,assume_utc,origin_wind_speed_sample_state_mean_ms,Late,6529,5,1.674,1.614,1.361,1.930



Scenarios were analyzed separately. These summaries do not establish causality or predictive value.


## 7. Export all results as CSV
Outputs go to `data/processed/` (combined datasets) and `outputs/tables/combined_features/` (EDA tables). Existing files with the same names are overwritten. Old CSV.GZ files are not deleted.

The combined files contain retrospective columns for EDA. Before purchase-time modeling, explicitly select eligible predictors; exclude actual delivery outcomes, post-purchase timestamps and derived delays from the inputs. Fit imputation and other preprocessing on training data only.


In [11]:
output_dir = ROOT / "data/processed"
report_dir = ROOT / "outputs/tables/combined_features"
output_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)

for scenario, df in combined_tables.items():
    path = output_dir / f"research_data_weather_price_holiday_{scenario}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved: {path.name} — {df.shape}")

reports = {
    "missingness_comparison": missingness_comparison,
    "price_summary": price_summary,
    "weather_summary": weather_summary,
    "window_holiday_summary": window_holiday_summary,
    "monthly_holiday_summary": monthly_holiday_summary,
}
for scenario, report in missingness_tables.items():
    reports[f"missingness_{scenario}"] = report
for name, report in holiday_summary_tables.items():
    reports[f"holiday_{name}"] = report
for name, report in price_quartile_tables.items():
    reports[f"quartiles_{name}"] = report

# Save the corrected calendar-day window summaries as well.
for group_col in ["purchase_holiday_group", "days_to_next_holiday_group",
                  "estimated_window_holiday_group"]:
    reports[f"window_by_{group_col}"] = (
        holiday_eda.groupby(group_col, observed=True)
        .agg(order_count=("order_id", "size"),
             mean_window_days=("estimated_window_days", "mean"),
             median_window_days=("estimated_window_days", "median"))
        .reset_index()
    )

for name, report in reports.items():
    report.to_csv(report_dir / f"{name}.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(reports)} report tables to: {report_dir}")
print("All exports are plain CSV. No missing values were imputed.")


Saved: research_data_weather_price_holiday_assume_sao_paulo.csv — (99441, 74)
Saved: research_data_weather_price_holiday_assume_utc.csv — (99441, 74)
Saved 17 report tables to: c:\Users\LENOVO\OneDrive\Desktop\PTTK\Group5_STAT3013.R11\outputs\tables\combined_features
All exports are plain CSV. No missing values were imputed.
